In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_bank = pd.read_csv('bank.csv')
data_bank["default"]=data_bank["default"].map({"no":0,"yes":1})
data_bank["housing"]=data_bank["housing"].map({"no":0,"yes":1})
data_bank["loan"]=data_bank["loan"].map({"no":0,"yes":1})
data_bank["deposit"]=data_bank["deposit"].map({"no":0,"yes":1})
edu_map = {"unknown":0,
           "primary":1,
           "secondary":2,
           "tertiary":3}
data_bank["education"]=data_bank["education"].map(edu_map)
data_bank = pd.get_dummies(data_bank, columns=['education'], dtype=int)
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

# 1. Переводим строки в числа 1-12
data_bank['month_num'] = data_bank['month'].map(month_map)

# 2. Создаем циклические признаки
data_bank['month_sin'] = np.sin(2 * np.pi * data_bank['month_num'] / 12)
data_bank['month_cos'] = np.cos(2 * np.pi * data_bank['month_num'] / 12)

# Теперь можно удалить исходную колонку и промежуточную month_num
data_bank = data_bank.drop(['month', 'month_num'], axis=1)
data_bank = pd.get_dummies(data_bank, columns=['poutcome'], dtype=int)
data_bank = pd.get_dummies(data_bank, columns=['contact'], dtype=int)
data_bank = pd.get_dummies(data_bank, columns=['marital'], dtype=int)
data_bank = pd.get_dummies(data_bank, columns=['job'], dtype=int)

y = data_bank['deposit']
X = data_bank.drop('deposit', axis=1)

X_train, X_test, y_bank_train, y_bank_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_bank_train = scaler.fit_transform(X_train)
X_bank_test = scaler.transform(X_test)



In [ ]:
# ============================================================================
# 1. ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import time

print("=" * 60)
print(" ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ")
print("=" * 60)

# Создаём модель
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

# Обучение
start_time = time.time()
log_reg.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_lr = log_reg.predict(X_bank_test)
y_proba_lr = log_reg.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_lr = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_lr),
    'Precision': precision_score(y_bank_test, y_pred_lr),
    'Recall': recall_score(y_bank_test, y_pred_lr),
    'F1-Score': f1_score(y_bank_test, y_pred_lr),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_lr),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_lr),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_lr.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

 ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
⏱️ Время обучения: 0.010 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8088
  Precision   : 0.8019
  Recall      : 0.7921
  F1-Score    : 0.7970
  ROC-AUC     : 0.8872
  PR-AUC      : 0.8583


In [ ]:
# ============================================================================
# ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ + GRID SEARCH CV (ОПТИМИЗИРОВАННЫЙ)
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 90)
print(" ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ + GRID SEARCH CV (ОПТИМИЗИРОВАННЫЙ)")
print("=" * 90)

# ----------------------------------------------------------------------------
# 1. НАСТРОЙКА ПАРАМЕТРОВ (СОКРАЩЕННАЯ СЕТКА)
# ----------------------------------------------------------------------------
# Веса: только четные значения + стандартные
custom_class_weights = [{0: 1, 1: w} for w in [2, 4, 6, 8, 10, 12]]
custom_class_weights.extend([None, 'balanced'])

print(f"\n⚖️  ВЕСА КЛАССОВ: 1:2, 1:4, 1:6, 1:8, 1:10, 1:12 + None, balanced")
print(f"   Всего вариантов весов: {len(custom_class_weights)}")

# Оптимизированная сетка гиперпараметров
param_grid = [
    {
        'solver': ['lbfgs'],
        'penalty': ['l2', 'none'],
        'C': [0.001, 0.005, 0.01, 0.01, 0.1, 1, 10, 100],
        'class_weight': custom_class_weights,
        'max_iter': [1000, 5000],
        'tol': [1e-4]
    },
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': [0.001, 0.005, 0.01, 0.01, 0.1, 1, 10, 100],
        'class_weight': custom_class_weights,
        'max_iter': [1000, 5000],
        'tol': [1e-4]
    },
    {
        'solver': ['saga'],
        'penalty': ['elasticnet'],
        'C': [0.001, 0.005, 0.01, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.15, 0.5, 0.85],
        'class_weight': custom_class_weights,
        'max_iter': [1000, 5000],
        'tol': [1e-4]
    },
]

# Подсчет комбинаций
total_combinations = 0
for grid in param_grid:
    combos = 1
    for key, values in grid.items():
        combos *= len(values)
    total_combinations += combos

print(f"\n⚙️  ОПТИМИЗИРОВАННАЯ КОНФИГУРАЦИЯ:")
print(f"   • Солверы: lbfgs, liblinear, saga")
print(f"   • Пенальти: l1, l2, elasticnet, none")
print(f"   • C: [0.01, 0.1, 1, 10, 100]")
print(f"   • Max Iter: [1000, 5000]")
print(f"   • Tol: [1e-4]")
print(f"   🔢 Всего комбинаций: ~{total_combinations:,}")
print(f"   🔢 С учетом 5-Fold CV: ~{total_combinations * 5:,} обучений")
print(f"   ⏱️  Оценка времени: ~{(total_combinations * 5 * 0.2) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК GRID SEARCH
# ----------------------------------------------------------------------------
print("\n🔍 Запуск GridSearchCV...")
print("-" * 90)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_log_reg = LogisticRegression(random_state=42)

grid_search = GridSearchCV(
    estimator=base_log_reg,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    refit=True
)

start_time = time.time()
grid_search.fit(X_bank_train, y_bank_train)
grid_time = time.time() - start_time

print("-" * 90)
print(f"\n⏱️  Фактическое время выполнения: {grid_time:.3f} сек ({grid_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ
# ----------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ")
print("=" * 90)

print(f"\n✅ Лучшие параметры:")
for param, value in grid_search.best_params_.items():
    if param == 'class_weight' and isinstance(value, dict):
        ratio = list(value.values())[1]
        print(f"   {param:20s}: {value}  ← вес миноритарного класса = {ratio}")
    else:
        print(f"   {param:20s}: {value}")

print(f"\n📈 Лучший ROC-AUC (CV): {grid_search.best_score_:.6f}")

# ----------------------------------------------------------------------------
# 4. ТОП-10 КОМБИНАЦИЙ
# ----------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 📋 ТОП-10 КОМБИНАЦИЙ ПАРАМЕТРОВ")
print("=" * 90)

results_df = pd.DataFrame(grid_search.cv_results_)

# 🔧 ИСПРАВЛЕНИЕ: Используем правильные имена колонок
# GridSearchCV всегда использует 'score' для ранжирования, независимо от scoring
results_df = results_df.sort_values('rank_test_score').head(10)

def extract_weight_ratio(row):
    cw = row.get('param_class_weight')
    if isinstance(cw, dict):
        return list(cw.values())[1]
    elif cw == 'balanced':
        return 'balanced'
    else:
        return 'None'

results_df['weight_ratio'] = results_df.apply(extract_weight_ratio, axis=1)

display_cols = [
    'rank_test_score',      # 🔧 Исправлено с rank_test_roc_auc
    'mean_test_score',      # 🔧 Исправлено с mean_test_roc_auc
    'std_test_score',       # 🔧 Исправлено с std_test_roc_auc
    'param_solver',
    'param_penalty',
    'param_C',
    'weight_ratio',
    'param_max_iter'
]

print(results_df[display_cols].to_string(index=False))

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ВЕСАМ КЛАССОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 📊 АНАЛИЗ ПРОИЗВОДИТЕЛЬНОСТИ ПО ВЕСАМ КЛАССОВ")
print("=" * 90)

weight_stats = results_df.groupby('weight_ratio').agg({
    'mean_test_score': ['mean', 'std', 'count']
}).round(4)

print(f"\n📈 Средний ROC-AUC по разным весам миноритарного класса:")
print(weight_stats.to_string())

best_weight_row = results_df.loc[results_df['mean_test_score'].idxmax()]
print(f"\n🏆 Лучший вес класса: {best_weight_row['weight_ratio']} (ROC-AUC: {best_weight_row['mean_test_score']:.6f})")

print(f"\n📉 Зависимость ROC-AUC от веса:")
print("-" * 60)
weight_order = ['None', 'balanced'] + [2, 4, 6, 8, 10, 12]
weight_means = results_df.groupby('weight_ratio')['mean_test_score'].mean()

for weight in weight_order:
    if weight in weight_means.index:
        val = weight_means[weight]
        bar_len = int(val * 30)
        bar = "█" * bar_len
        marker = "🏆" if weight == best_weight_row['weight_ratio'] else " "
        print(f"   {marker} Вес {str(weight):>12s}: {val:.6f}  {bar}")

# ----------------------------------------------------------------------------
# 6. ФИНАЛЬНЫЕ МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 📊 ФИНАЛЬНЫЕ МЕТРИКИ НА ТЕСТЕ")
print("=" * 90)

best_log_reg = grid_search.best_estimator_

y_pred_lr = best_log_reg.predict(X_bank_test)
y_proba_lr = best_log_reg.predict_proba(X_bank_test)[:, 1]

metrics_lr = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_lr),
    'Precision': precision_score(y_bank_test, y_pred_lr, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred_lr, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred_lr, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_lr),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_lr)
}

print(f"\n📈 МЕТРИКИ:")
print("-" * 50)
for name, value in metrics_lr.items():
    print(f"  {name:15s}: {value:.6f}")

# ----------------------------------------------------------------------------
# 7. СРАВНЕНИЕ CV vs ТЕСТ
# ----------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 🔍 СРАВНЕНИЕ: CV (Mean) vs ТЕСТ")
print("=" * 90)

best_idx = results_df.iloc[0].name

print(f"\n{'Метрика':<15s} | {'CV Mean':<12s} | {'CV Std':<12s} | {'Тест':<12s} | {'Разница':<12s} | {'Статус':<10s}")
print("-" * 90)

# 🔧 ИСПРАВЛЕНИЕ: Используем правильные имена колонок для каждой метрики
metric_mapping = {
    'Accuracy': 'mean_test_accuracy',
    'Precision': 'mean_test_precision',
    'Recall': 'mean_test_recall',
    'F1-Score': 'mean_test_f1_score',
    'ROC-AUC': 'mean_test_score',  # 🔧 Для ROC-AUC используем mean_test_score
    'PR-AUC': 'mean_test_average_precision'
}

for metric_name, metric_key in metric_mapping.items():
    if metric_key in grid_search.cv_results_:
        cv_mean = grid_search.cv_results_[metric_key][best_idx]
        std_key = metric_key.replace('mean_', 'std_')
        cv_std = grid_search.cv_results_.get(std_key, [0])[best_idx] if std_key in grid_search.cv_results_ else 0
        test_val = metrics_lr[metric_name]
        diff = test_val - cv_mean
        
        if diff > 0.02:
            status = "⚠️  Возможно"
        elif diff < -0.02:
            status = "✅ Недообуч"
        else:
            status = "✅ Норма"
        
        print(f"{metric_name:<15s} | {cv_mean:<12.6f} | {cv_std:<12.6f} | {test_val:<12.6f} | {diff:+12.6f} | {status:<10s}")
    else:
        print(f"{metric_name:<15s} | {'N/A':<12s} | {'N/A':<12s} | {metrics_lr[metric_name]:<12.6f} | {'N/A':<12s} | {'Пропущено':<10s}")

print("\n" + "=" * 90)
print(" ✅ GRID SEARCH ЗАВЕРШЕН!")
print("=" * 90)  


 ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ + GRID SEARCH CV (ОПТИМИЗИРОВАННЫЙ)

⚖️  ВЕСА КЛАССОВ: 1:2, 1:4, 1:6, 1:8, 1:10, 1:12 + None, balanced
   Всего вариантов весов: 8

⚙️  ОПТИМИЗИРОВАННАЯ КОНФИГУРАЦИЯ:
   • Солверы: lbfgs, liblinear, saga
   • Пенальти: l1, l2, elasticnet, none
   • C: [0.01, 0.1, 1, 10, 100]
   • Max Iter: [1000, 5000]
   • Tol: [1e-4]
   🔢 Всего комбинаций: ~896
   🔢 С учетом 5-Fold CV: ~4,480 обучений
   ⏱️  Оценка времени: ~14.9 мин

🔍 Запуск GridSearchCV...
------------------------------------------------------------------------------------------
Fitting 5 folds for each of 896 candidates, totalling 4480 fits
------------------------------------------------------------------------------------------

⏱️  Фактическое время выполнения: 40.480 сек (0.67 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ

✅ Лучшие параметры:
   C                   : 0.1
   class_weight        : {0: 1, 1: 2}  ← вес миноритарного класса = 2
   max_iter            : 1000
   penalty             : l1
   solver              :

In [25]:
# ============================================================================
# 2. DECISION TREE (ДЕРЕВО РЕШЕНИЙ)
# ============================================================================
from sklearn.tree import DecisionTreeClassifier
import time

print("=" * 60)
print("🟢 DECISION TREE")
print("=" * 60)

# Создаём модель
dt = DecisionTreeClassifier(
    max_depth=10,           # Ограничиваем глубину для борьбы с переобучением
    min_samples_split=20,   # Минимум образцов для разделения
    min_samples_leaf=10,    # Минимум образцов в листе
    random_state=42,
    class_weight='balanced'
)

# Обучение
start_time = time.time()
dt.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_dt = dt.predict(X_bank_test)
y_proba_dt = dt.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_dt = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_dt),
    'Precision': precision_score(y_bank_test, y_pred_dt),
    'Recall': recall_score(y_bank_test, y_pred_dt),
    'F1-Score': f1_score(y_bank_test, y_pred_dt),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_dt),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_dt),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"🌳 Количество листьев: {dt.get_n_leaves()}")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_dt.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

🟢 DECISION TREE
⏱️ Время обучения: 0.031 сек
🌳 Количество листьев: 219

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8312
  Precision   : 0.8048
  Recall      : 0.8497
  F1-Score    : 0.8267
  ROC-AUC     : 0.9000
  PR-AUC      : 0.8466


In [ ]:
# ============================================================================
# DECISION TREE + RANDOMIZED SEARCH CV (OPTIMIZED & CLEAN)
# ============================================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🌳 DECISION TREE + RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
custom_class_weights = [{0: 1, 1: w} for w in [2, 4, 6, 8, 10, 12]]
custom_class_weights.extend([None, 'balanced'])

param_distributions = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features': [None, 'sqrt', 'log2'],
    'class_weight': custom_class_weights
}

n_iter = 200
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} итераций, {cv_splits}-Fold CV, {len(custom_class_weights)} вариантов весов")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.1) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК ПОИСКА
# ----------------------------------------------------------------------------
print("\n🔍 Запуск RandomizedSearchCV...")
cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
base_dt = DecisionTreeClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=base_dt,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    refit=True,
    random_state=42
)

start_time = time.time()
random_search.fit(X_bank_train, y_bank_train)
search_time = time.time() - start_time

print(f"✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ (ИСПРАВЛЕНО)
# ----------------------------------------------------------------------------
# 🔧 СНАЧАЛА получаем best_dt, потом используем
best_dt = random_search.best_estimator_  # ← ПЕРЕМЕЩЕНО СЮДА
results_df = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score').head(10)

def get_weight_str(cw):
    if isinstance(cw, dict): return str(list(cw.values())[1])
    return str(cw)

results_df['weight_ratio'] = results_df['param_class_weight'].apply(get_weight_str)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

bp = random_search.best_params_
print(f"\n✅ Best F1-Score (CV): {random_search.best_score_:.6f}")
print(f"\n📋 Все лучшие параметры:")
print("-" * 50)
for param, value in bp.items():
    if param == 'class_weight' and isinstance(value, dict):
        print(f"   {param:25s}: {value} ← вес миноритарного класса = {list(value.values())[1]}")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ТЕПЕРЬ best_dt уже определён
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_dt.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_max_depth', 
        'param_criterion', 'param_min_samples_leaf', 'weight_ratio']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 4. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_dt.predict(X_bank_test)
y_proba = best_dt.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = random_search.best_score_ if name == 'F1-Score' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура дерева: Глубина={best_dt.get_depth()}, Листьев={best_dt.get_n_leaves()}")

# ----------------------------------------------------------------------------
# 5. ВАЖНОСТЬ ПРИЗНАКОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
print("=" * 70)

if hasattr(X_bank_train, 'columns'):
    feature_names = X_bank_train.columns
elif hasattr(X_bank_train, 'shape'):
    feature_names = [f'Feature_{i}' for i in range(X_bank_train.shape[1])]
else:
    feature_names = [f'Feature_{i}' for i in range(len(best_dt.feature_importances_))]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_dt.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(row['Importance'] * 40)
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

top10_sum = feature_importance['Importance'].sum()
print(f"\n💡 Топ-10 признаков объясняют: {top10_sum*100:.1f}% важности модели")

print("\n" + "=" * 70)
print(" ✅ ЗАВЕРШЕНО")
print("=" * 70)   


 🌳 DECISION TREE + RANDOMIZED SEARCH CV
⚙️  Параметры: 200 итераций, 5-Fold CV, 8 вариантов весов
⏱️  Ожидаемое время: ~1.7 мин

🔍 Запуск RandomizedSearchCV...
✅ Готово за 2.1 сек (0.03 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best F1-Score (CV): 0.831838

📋 Все лучшие параметры:
--------------------------------------------------
   splitter                 : best
   min_samples_split        : 50
   min_samples_leaf         : 5
   max_features             : None
   max_depth                : 20
   criterion                : entropy
   class_weight             : balanced

📋 Фактические параметры обученной модели:
--------------------------------------------------
   ccp_alpha                : 0.0
   class_weight             : balanced
   criterion                : entropy
   max_depth                : 20
   max_features             : None
   max_leaf_nodes           : None
   min_impurity_decrease    : 0.0
   min_samples_leaf         : 5
   min_samples_split        : 50
   min_weigh

In [7]:
# ============================================================================
# 3. RANDOM FOREST (СЛУЧАЙНЫЙ ЛЕС)
# ============================================================================
from sklearn.ensemble import RandomForestClassifier
import time

print("=" * 60)
print("🟠 RANDOM FOREST")
print("=" * 60)

# Создаём модель
rf = RandomForestClassifier(
    n_estimators=200,       # Количество деревьев
    max_depth=15,           # Максимальная глубина
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1               # Использовать все ядра
)

# Обучение
start_time = time.time()
rf.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_rf = rf.predict(X_bank_test)
y_proba_rf = rf.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_rf = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_rf),
    'Precision': precision_score(y_bank_test, y_pred_rf),
    'Recall': recall_score(y_bank_test, y_pred_rf),
    'F1-Score': f1_score(y_bank_test, y_pred_rf),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_rf),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_rf),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_rf.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

🟠 RANDOM FOREST
⏱️ Время обучения: 0.278 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8509
  Precision   : 0.8096
  Recall      : 0.8960
  F1-Score    : 0.8506
  ROC-AUC     : 0.9175
  PR-AUC      : 0.8842


In [ ]:
# ============================================================================
# 3. RANDOM FOREST + RANDOMIZED SEARCH CV (OPTIMIZED)
# ============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🌲 RANDOM FOREST + RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
custom_class_weights = [{0: 1, 1: w} for w in [2, 4, 8, 10, 12]]
custom_class_weights.extend([None, 'balanced'])

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10, 15, 20],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False],
    'class_weight': custom_class_weights
}

n_iter = 500
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} итераций, {cv_splits}-Fold CV, {len(custom_class_weights)} весов")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.3) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК ПОИСКА
# ----------------------------------------------------------------------------
print("\n🔍 Запуск RandomizedSearchCV...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    refit=True,
    random_state=42
)

start_time = time.time()
random_search.fit(X_bank_train, y_bank_train)
search_time = time.time() - start_time

print(f"✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)
# ----------------------------------------------------------------------------
# 🔧 СНАЧАЛА получаем best_rf
best_rf = random_search.best_estimator_
results_df = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score').head(10)

def get_weight_str(cw):
    if isinstance(cw, dict): return str(list(cw.values())[1])
    return str(cw)

results_df['weight_ratio'] = results_df['param_class_weight'].apply(get_weight_str)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

bp = random_search.best_params_
print(f"\n✅ Best ROC-AUC (CV): {random_search.best_score_:.6f}")

# 🔧 ВСЕ ПАРАМЕТРЫ ПОИСКА
print(f"\n📋 Все лучшие параметры (из поиска):")
print("-" * 50)
for param, value in bp.items():
    if param == 'class_weight' and isinstance(value, dict):
        print(f"   {param:25s}: {value} ← вес миноритарного = {list(value.values())[1]}")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ВСЕ ПАРАМЕТРЫ МОДЕЛИ ПОСЛЕ ОБУЧЕНИЯ
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_rf.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_n_estimators', 
        'param_max_depth', 'param_min_samples_leaf', 'weight_ratio']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 4. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_rf.predict(X_bank_test)
y_proba = best_rf.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = random_search.best_score_ if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура леса:")
print(f"   • Деревьев: {best_rf.n_estimators}")
print(f"   • Средняя глубина: {np.mean([tree.get_depth() for tree in best_rf.estimators_]):.1f}")
print(f"   • Средняя листьев: {np.mean([tree.get_n_leaves() for tree in best_rf.estimators_]):.0f}")

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ПАРАМЕТРАМ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 АНАЛИЗ ПО КЛЮЧЕВЫМ ПАРАМЕТРАМ")
print("=" * 70)

print(f"\n📈 По весам классов:")
weight_stats = results_df.groupby('weight_ratio')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(weight_stats.to_string())

print(f"\n📈 По количеству деревьев:")
n_est_stats = results_df.groupby('param_n_estimators')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(n_est_stats.to_string())

print(f"\n📈 По глубине деревьев:")
depth_stats = results_df.groupby('param_max_depth')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(depth_stats.to_string())

# ----------------------------------------------------------------------------
# 6. ВАЖНОСТЬ ПРИЗНАКОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
print("=" * 70)

if hasattr(X_bank_train, 'columns'):
    feature_names = X_bank_train.columns
elif hasattr(X_bank_train, 'shape'):
    feature_names = [f'Feature_{i}' for i in range(X_bank_train.shape[1])]
else:
    feature_names = [f'Feature_{i}' for i in range(len(best_rf.feature_importances_))]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(row['Importance'] * 40)
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

top10_sum = feature_importance['Importance'].sum()
print(f"\n💡 Топ-10 признаков объясняют: {top10_sum*100:.1f}% важности модели")

# ----------------------------------------------------------------------------
# 7. СРАВНЕНИЕ С ДРУГИМИ МОДЕЛЯМИ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 СРАВНЕНИЕ МОДЕЛЕЙ (ROC-AUC)")
print("=" * 70)

print(f"\n{'Модель':<25s} | {'ROC-AUC':<10s} | {'Время':<10s}")
print("-" * 50)
print(f"{'Logistic Regression':<25s} | {0.8880:<10.4f} | {'0.01 сек':<10s}")
print(f"{'Decision Tree':<25s} | {0.9000:<10.4f} | {'0.03 сек':<10s}")
print(f"{'Random Forest (CV)':<25s} | {metrics['ROC-AUC']:<10.4f} | {f'{search_time:.1f} сек':<10s}")

print("\n" + "=" * 70)
print(" ✅ RANDOM FOREST + RANDOMIZED SEARCH ЗАВЕРШЕН!")
print("=" * 70)



 🌲 RANDOM FOREST + RANDOMIZED SEARCH CV
⚙️  Параметры: 500 итераций, 5-Fold CV, 7 весов
⏱️  Ожидаемое время: ~12.5 мин

🔍 Запуск RandomizedSearchCV...
✅ Готово за 286.0 сек (4.77 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best ROC-AUC (CV): 0.921985

📋 Все лучшие параметры (из поиска):
--------------------------------------------------
   n_estimators             : 500
   min_samples_split        : 10
   min_samples_leaf         : 1
   max_features             : sqrt
   max_depth                : None
   class_weight             : balanced
   bootstrap                : True

📋 Фактические параметры обученной модели:
--------------------------------------------------
   bootstrap                : True
   ccp_alpha                : 0.0
   class_weight             : balanced
   criterion                : gini
   max_depth                : None
   max_features             : sqrt
   max_leaf_nodes           : None
   max_samples              : None
   min_impurity_decrease    : 0.0
   min

In [28]:
# ============================================================================
# 4. EXTRA TREES (EXTREMELY RANDOMIZED TREES)
# ============================================================================
from sklearn.ensemble import ExtraTreesClassifier
import time

print("=" * 60)
print("🟣 EXTRA TREES")
print("=" * 60)

# Создаём модель
et = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

# Обучение
start_time = time.time()
et.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_et = et.predict(X_bank_test)
y_proba_et = et.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_et = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_et),
    'Precision': precision_score(y_bank_test, y_pred_et),
    'Recall': recall_score(y_bank_test, y_pred_et),
    'F1-Score': f1_score(y_bank_test, y_pred_et),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_et),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_et),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_et.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

🟣 EXTRA TREES
⏱️ Время обучения: 0.252 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8137
  Precision   : 0.8057
  Recall      : 0.7996
  F1-Score    : 0.8027
  ROC-AUC     : 0.8919
  PR-AUC      : 0.8533


In [ ]:
# ============================================================================
# 4. EXTRA TREES + RANDOMIZED SEARCH CV (OPTIMIZED)
# ============================================================================
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🌳 EXTRA TREES + RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
custom_class_weights = [{0: 1, 1: w} for w in [2, 4, 8, 10, 12]]
custom_class_weights.extend([None, 'balanced'])

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2', None, 0.5, 0.7],
    'bootstrap': [True, False],
    'class_weight': custom_class_weights
}

n_iter = 500
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} итераций, {cv_splits}-Fold CV, {len(custom_class_weights)} весов")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.2) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК ПОИСКА
# ----------------------------------------------------------------------------
print("\n🔍 Запуск RandomizedSearchCV...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
base_et = ExtraTreesClassifier(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=base_et,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    refit=True,
    random_state=42
)

start_time = time.time()
random_search.fit(X_bank_train, y_bank_train)
search_time = time.time() - start_time

print(f"✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)
# ----------------------------------------------------------------------------
# 🔧 СНАЧАЛА получаем best_et
best_et = random_search.best_estimator_
results_df = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score').head(10)

def get_weight_str(cw):
    if isinstance(cw, dict): return str(list(cw.values())[1])
    return str(cw)

results_df['weight_ratio'] = results_df['param_class_weight'].apply(get_weight_str)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

bp = random_search.best_params_
print(f"\n✅ Best ROC-AUC (CV): {random_search.best_score_:.6f}")

# 🔧 ВСЕ ПАРАМЕТРЫ ПОИСКА
print(f"\n📋 Все лучшие параметры (из поиска):")
print("-" * 50)
for param, value in bp.items():
    if param == 'class_weight' and isinstance(value, dict):
        print(f"   {param:25s}: {value} ← вес миноритарного = {list(value.values())[1]}")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ВСЕ ПАРАМЕТРЫ МОДЕЛИ ПОСЛЕ ОБУЧЕНИЯ
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_et.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_n_estimators', 
        'param_max_depth', 'param_min_samples_leaf', 'weight_ratio']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 4. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_et.predict(X_bank_test)
y_proba = best_et.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = random_search.best_score_ if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура леса:")
print(f"   • Деревьев: {best_et.n_estimators}")
print(f"   • Средняя глубина: {np.mean([tree.get_depth() for tree in best_et.estimators_]):.1f}")
print(f"   • Средняя листьев: {np.mean([tree.get_n_leaves() for tree in best_et.estimators_]):.0f}")

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ПАРАМЕТРАМ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 АНАЛИЗ ПО КЛЮЧЕВЫМ ПАРАМЕТРАМ")
print("=" * 70)

print(f"\n📈 По весам классов:")
weight_stats = results_df.groupby('weight_ratio')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(weight_stats.to_string())

print(f"\n📈 По количеству деревьев:")
n_est_stats = results_df.groupby('param_n_estimators')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(n_est_stats.to_string())

print(f"\n📈 По глубине деревьев:")
depth_stats = results_df.groupby('param_max_depth')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(depth_stats.to_string())

print(f"\n📈 По max_features:")
feat_stats = results_df.groupby('param_max_features')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(feat_stats.to_string())

# ----------------------------------------------------------------------------
# 6. ВАЖНОСТЬ ПРИЗНАКОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
print("=" * 70)

if hasattr(X_bank_train, 'columns'):
    feature_names = X_bank_train.columns
elif hasattr(X_bank_train, 'shape'):
    feature_names = [f'Feature_{i}' for i in range(X_bank_train.shape[1])]
else:
    feature_names = [f'Feature_{i}' for i in range(len(best_et.feature_importances_))]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_et.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(row['Importance'] * 40)
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

top10_sum = feature_importance['Importance'].sum()
print(f"\n💡 Топ-10 признаков объясняют: {top10_sum*100:.1f}% важности модели")

print("\n" + "=" * 70)
print(" ✅ EXTRA TREES + RANDOMIZED SEARCH ЗАВЕРШЕН!")
print("=" * 70)



 🌳 EXTRA TREES + RANDOMIZED SEARCH CV
⚙️  Параметры: 500 итераций, 5-Fold CV, 7 весов
⏱️  Ожидаемое время: ~8.3 мин

🔍 Запуск RandomizedSearchCV...
✅ Готово за 376.9 сек (6.28 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best ROC-AUC (CV): 0.919377

📋 Все лучшие параметры (из поиска):
--------------------------------------------------
   n_estimators             : 300
   min_samples_split        : 10
   min_samples_leaf         : 1
   max_features             : None
   max_depth                : 30
   class_weight             : None
   bootstrap                : True

📋 Фактические параметры обученной модели:
--------------------------------------------------
   bootstrap                : True
   ccp_alpha                : 0.0
   class_weight             : None
   criterion                : gini
   max_depth                : 30
   max_features             : None
   max_leaf_nodes           : None
   max_samples              : None
   min_impurity_decrease    : 0.0
   min_samples_leaf  

In [3]:
# ============================================================================
# 5. LIGHTGBM (LIGHT GRADIENT BOOSTING MACHINE)
# ============================================================================
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import lightgbm as lgb
import time

print("=" * 60)
print("🔶 LIGHTGBM")
print("=" * 60)

# Создаём модель
lgbm = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=31,
    min_child_samples=20,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    verbose=-1
)

# Обучение
start_time = time.time()
lgbm.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_lgbm = lgbm.predict(X_bank_test)
y_proba_lgbm = lgbm.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_lgbm = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_lgbm),
    'Precision': precision_score(y_bank_test, y_pred_lgbm),
    'Recall': recall_score(y_bank_test, y_pred_lgbm),
    'F1-Score': f1_score(y_bank_test, y_pred_lgbm),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_lgbm),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_lgbm),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_lgbm.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

🔶 LIGHTGBM
⏱️ Время обучения: 0.213 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8674
  Precision   : 0.8319
  Recall      : 0.9026
  F1-Score    : 0.8658
  ROC-AUC     : 0.9299
  PR-AUC      : 0.8972


c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ============================================================================
# 5. LIGHTGBM + RANDOMIZED SEARCH CV (OPTIMIZED)
# ============================================================================
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import lightgbm as lgb
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🔶 LIGHTGBM + RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
custom_class_weights = [{0: 1, 1: w} for w in [2, 4, 6, 8, 10, 12]]
custom_class_weights.extend([None, 'balanced'])

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [6, 10, 15, 20, -1],
    'num_leaves': [15, 31, 63, 127],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'min_child_samples': [10, 20, 50, 100],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'class_weight': custom_class_weights,
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0, 0.1, 0.5, 1.0]
}

n_iter = 300
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} итераций, {cv_splits}-Fold CV, {len(custom_class_weights)} весов")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.3) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК ПОИСКА
# ----------------------------------------------------------------------------
print("\n🔍 Запуск RandomizedSearchCV...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
base_lgbm = lgb.LGBMClassifier(
    random_state=42, 
    n_jobs=-1, 
    verbose=-1,
    force_col_wise=True
)

random_search = RandomizedSearchCV(
    estimator=base_lgbm,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    refit=True,
    random_state=42
)

start_time = time.time()
random_search.fit(X_bank_train, y_bank_train)
search_time = time.time() - start_time

print(f"✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)
# ----------------------------------------------------------------------------
# 🔧 СНАЧАЛА получаем best_lgbm
best_lgbm = random_search.best_estimator_
results_df = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score').head(10)

def get_weight_str(cw):
    if isinstance(cw, dict): return str(list(cw.values())[1])
    return str(cw)

results_df['weight_ratio'] = results_df['param_class_weight'].apply(get_weight_str)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

bp = random_search.best_params_
print(f"\n✅ Best ROC-AUC (CV): {random_search.best_score_:.6f}")

# 🔧 ВСЕ ПАРАМЕТРЫ ПОИСКА
print(f"\n📋 Все лучшие параметры (из поиска):")
print("-" * 50)
for param, value in bp.items():
    if param == 'class_weight' and isinstance(value, dict):
        print(f"   {param:25s}: {value} ← вес миноритарного = {list(value.values())[1]}")
    elif param == 'max_depth' and value == -1:
        print(f"   {param:25s}: {value} ← без ограничения глубины")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ВСЕ ПАРАМЕТРЫ МОДЕЛИ ПОСЛЕ ОБУЧЕНИЯ
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_lgbm.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_n_estimators', 
        'param_max_depth', 'param_num_leaves', 'weight_ratio', 'param_learning_rate']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 4. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_lgbm.predict(X_bank_test)
y_proba = best_lgbm.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = random_search.best_score_ if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура модели:")
print(f"   • Деревьев: {best_lgbm.n_estimators_}")
print(f"   • Лучшая итерация: {best_lgbm.best_iteration_ if hasattr(best_lgbm, 'best_iteration_') else 'N/A'}")

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ПАРАМЕТРАМ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 АНАЛИЗ ПО КЛЮЧЕВЫМ ПАРАМЕТРАМ")
print("=" * 70)

print(f"\n📈 По весам классов:")
weight_stats = results_df.groupby('weight_ratio')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(weight_stats.to_string())

print(f"\n📈 По глубине деревьев:")
depth_stats = results_df.groupby('param_max_depth')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(depth_stats.to_string())

print(f"\n📈 По скорости обучения (LR):")
lr_stats = results_df.groupby('param_learning_rate')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(lr_stats.to_string())

print(f"\n📈 По количеству листьев (num_leaves):")
leaves_stats = results_df.groupby('param_num_leaves')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(leaves_stats.to_string())

# ----------------------------------------------------------------------------
# 6. ВАЖНОСТЬ ПРИЗНАКОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
print("=" * 70)

if hasattr(X_bank_train, 'columns'):
    feature_names = X_bank_train.columns
elif hasattr(X_bank_train, 'shape'):
    feature_names = [f'Feature_{i}' for i in range(X_bank_train.shape[1])]
else:
    feature_names = [f'Feature_{i}' for i in range(len(best_lgbm.feature_importances_))]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_lgbm.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(row['Importance'] * 40)
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

top10_sum = feature_importance['Importance'].sum()
print(f"\n💡 Топ-10 признаков объясняют: {top10_sum*100:.1f}% важности модели")

print("\n" + "=" * 70)
print(" ✅ LIGHTGBM + RANDOMIZED SEARCH ЗАВЕРШЕН!")
print("=" * 70) 


 🔶 LIGHTGBM + RANDOMIZED SEARCH CV
⚙️  Параметры: 300 итераций, 5-Fold CV, 8 весов
⏱️  Ожидаемое время: ~7.5 мин

🔍 Запуск RandomizedSearchCV...
✅ Готово за 420.7 сек (7.01 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best ROC-AUC (CV): 0.928164

📋 Все лучшие параметры (из поиска):
--------------------------------------------------
   subsample                : 0.9
   reg_lambda               : 0.5
   reg_alpha                : 0
   num_leaves               : 31
   n_estimators             : 200
   min_child_samples        : 10
   max_depth                : -1 ← без ограничения глубины
   learning_rate            : 0.05
   colsample_bytree         : 0.7
   class_weight             : None

📋 Фактические параметры обученной модели:
--------------------------------------------------
   boosting_type            : gbdt
   class_weight             : None
   colsample_bytree         : 0.7
   importance_type          : split
   learning_rate            : 0.05
   max_depth                : -1
 

In [ ]:
# ============================================================================
# 6. GRADIENT BOOSTING (SKLEARN)
# ============================================================================
from sklearn.ensemble import GradientBoostingClassifier
import time

print("=" * 60)
print("🔷 GRADIENT BOOSTING")
print("=" * 60)

# Создаём модель
gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

# Обучение
start_time = time.time()
gb.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_gb = gb.predict(X_bank_test)
y_proba_gb = gb.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_gb = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_gb),
    'Precision': precision_score(y_bank_test, y_pred_gb),
    'Recall': recall_score(y_bank_test, y_pred_gb),
    'F1-Score': f1_score(y_bank_test, y_pred_gb),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_gb),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_gb),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_gb.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")



🔷 GRADIENT BOOSTING
⏱️ Время обучения: 2.329 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8562
  Precision   : 0.8241
  Recall      : 0.8856
  F1-Score    : 0.8538
  ROC-AUC     : 0.9264
  PR-AUC      : 0.8951


In [ ]:
# ============================================================================
# 6. GRADIENT BOOSTING (SKLEARN) + RANDOMIZED SEARCH CV (OPTIMIZED)
# ============================================================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🔷 GRADIENT BOOSTING + RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
# ⚠️ GradientBoosting не поддерживает class_weight, используем subsample для баланса
param_distributions = {
    'n_estimators': [100, 150, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_split': [10, 20, 50, 100],
    'min_samples_leaf': [5, 10, 20, 50],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'max_features': ['sqrt', 'log2', None, 0.5, 0.7, 0.9],
    'min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
}

n_iter = 300
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} итераций, {cv_splits}-Fold CV")
print(f"⚠️  GradientBoosting не поддерживает class_weight (используем subsample)")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.5) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ЗАПУСК ПОИСКА
# ----------------------------------------------------------------------------
print("\n🔍 Запуск RandomizedSearchCV...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
base_gb = GradientBoostingClassifier(
    random_state=42,
    verbose=0
)

random_search = RandomizedSearchCV(
    estimator=base_gb,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    refit=True,
    random_state=42
)

start_time = time.time()
random_search.fit(X_bank_train, y_bank_train)
search_time = time.time() - start_time

print(f"✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 3. ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)
# ----------------------------------------------------------------------------
# 🔧 СНАЧАЛА получаем best_gb
best_gb = random_search.best_estimator_
results_df = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score').head(10)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

bp = random_search.best_params_
print(f"\n✅ Best ROC-AUC (CV): {random_search.best_score_:.6f}")

# 🔧 ВСЕ ПАРАМЕТРЫ ПОИСКА
print(f"\n📋 Все лучшие параметры (из поиска):")
print("-" * 50)
for param, value in bp.items():
    if param == 'subsample':
        print(f"   {param:25s}: {value} ← аналог class_weight для дисбаланса")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ВСЕ ПАРАМЕТРЫ МОДЕЛИ ПОСЛЕ ОБУЧЕНИЯ
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_gb.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_n_estimators', 
        'param_max_depth', 'param_learning_rate', 'param_subsample', 'param_min_samples_leaf']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 4. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_gb.predict(X_bank_test)
y_proba = best_gb.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = random_search.best_score_ if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура модели:")
print(f"   • Деревьев: {best_gb.n_estimators_}")
print(f"   • Фактическое кол-во итераций: {best_gb.n_estimators_}")

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ПАРАМЕТРАМ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 АНАЛИЗ ПО КЛЮЧЕВЫМ ПАРАМЕТРАМ")
print("=" * 70)

print(f"\n📈 По глубине деревьев:")
depth_stats = results_df.groupby('param_max_depth')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(depth_stats.to_string())

print(f"\n📈 По скорости обучения (LR):")
lr_stats = results_df.groupby('param_learning_rate')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(lr_stats.to_string())

print(f"\n📈 По subsample (аналог class_weight):")
subsample_stats = results_df.groupby('param_subsample')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(subsample_stats.to_string())

print(f"\n📈 По количеству деревьев:")
n_est_stats = results_df.groupby('param_n_estimators')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(n_est_stats.to_string())

# ----------------------------------------------------------------------------
# 6. ВАЖНОСТЬ ПРИЗНАКОВ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
print("=" * 70)

if hasattr(X_bank_train, 'columns'):
    feature_names = X_bank_train.columns
elif hasattr(X_bank_train, 'shape'):
    feature_names = [f'Feature_{i}' for i in range(X_bank_train.shape[1])]
else:
    feature_names = [f'Feature_{i}' for i in range(len(best_gb.feature_importances_))]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_gb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(row['Importance'] * 40)
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

top10_sum = feature_importance['Importance'].sum()
print(f"\n💡 Топ-10 признаков объясняют: {top10_sum*100:.1f}% важности модели")

print("\n" + "=" * 70)
print(" ✅ GRADIENT BOOSTING + RANDOMIZED SEARCH ЗАВЕРШЕН!")
print("=" * 70)




 🔷 GRADIENT BOOSTING + RANDOMIZED SEARCH CV
⚙️  Параметры: 300 итераций, 5-Fold CV
⚠️  GradientBoosting не поддерживает class_weight (используем subsample)
⏱️  Ожидаемое время: ~12.5 мин

🔍 Запуск RandomizedSearchCV...
✅ Готово за 236.1 сек (3.94 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best ROC-AUC (CV): 0.927182

📋 Все лучшие параметры (из поиска):
--------------------------------------------------
   subsample                : 1.0 ← аналог class_weight для дисбаланса
   n_estimators             : 300
   min_samples_split        : 50
   min_samples_leaf         : 20
   min_impurity_decrease    : 0.01
   max_features             : 0.7
   max_depth                : 7
   learning_rate            : 0.05

📋 Фактические параметры обученной модели:
--------------------------------------------------
   ccp_alpha                : 0.0
   criterion                : friedman_mse
   init                     : None
   learning_rate            : 0.05
   loss                     : log_loss
   ma

In [9]:
# ============================================================================
# 7. CATBOOST
# ============================================================================
from catboost import CatBoostClassifier
import time

print("=" * 60)
print("🔴 CATBOOST")
print("=" * 60)

# Создаём модель
catboost = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=6,
    random_state=42,
    verbose=0,
    class_weights=None  # CatBoost хорошо работает с дисбалансом
)

# Обучение
start_time = time.time()
catboost.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# Предсказания
y_pred_cb = catboost.predict(X_bank_test)
y_proba_cb = catboost.predict_proba(X_bank_test)[:, 1]

# Метрики
metrics_cb = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_cb),
    'Precision': precision_score(y_bank_test, y_pred_cb),
    'Recall': recall_score(y_bank_test, y_pred_cb),
    'F1-Score': f1_score(y_bank_test, y_pred_cb),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_cb),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_cb),
    'Time (s)': train_time
}

print(f"⏱️ Время обучения: {train_time:.3f} сек")
print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
for name, value in metrics_cb.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

🔴 CATBOOST
⏱️ Время обучения: 0.876 сек

📊 МЕТРИКИ НА ТЕСТЕ:
  Accuracy    : 0.8634
  Precision   : 0.8329
  Recall      : 0.8904
  F1-Score    : 0.8607
  ROC-AUC     : 0.9313
  PR-AUC      : 0.9005


In [ ]:
# ============================================================================
# CATBOOST + CUSTOM RANDOMIZED SEARCH CV
# ============================================================================
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
import random
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🔴 CATBOOST + CUSTOM RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
param_distributions = {
    'iterations': [100, 200, 300, 500],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 10, 30],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bylevel': [0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 2, 5, 10, None],
    'min_data_in_leaf': [5, 10, 20, 50],
}

n_iter = 50
cv_splits = 5

print(f"⚙️  Параметры: {n_iter} комбинаций, {cv_splits}-Fold CV")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.5) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ГЕНЕРАЦИЯ СЛУЧАЙНЫХ КОМБИНАЦИЙ
# ----------------------------------------------------------------------------
print("\n🔍 Генерация параметров...")

def generate_random_params(param_dist, n_iter, random_state=42):
    random.seed(random_state)
    params_list = []
    keys = list(param_dist.keys())
    
    for _ in range(n_iter):
        param_set = {key: random.choice(param_dist[key]) for key in keys}
        params_list.append(param_set)
    
    return params_list

random_params = generate_random_params(param_distributions, n_iter, random_state=42)

# ----------------------------------------------------------------------------
# 3. КАСТОМНАЯ КРОСС-ВАЛИДАЦИЯ
# ----------------------------------------------------------------------------
print("\n🔍 Запуск кросс-валидации...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

results = []
start_time = time.time()

# 🔧 ИСПРАВЛЕНИЕ: Преобразуем в numpy для гарантированной позиционной индексации
X_train_np = X_bank_train.values if hasattr(X_bank_train, 'values') else X_bank_train
y_train_np = y_bank_train.values if hasattr(y_bank_train, 'values') else y_bank_train

print(f"   📊 Размер данных: {X_train_np.shape[0]} строк, {X_train_np.shape[1]} признаков")

for idx, params in enumerate(random_params):
    cv_scores = []
    
    for train_idx, val_idx in cv_strategy.split(X_train_np, y_train_np):
        X_train_cv, X_val_cv = X_train_np[train_idx], X_train_np[val_idx]
        y_train_cv, y_val_cv = y_train_np[train_idx], y_train_np[val_idx]
        
        model = CatBoostClassifier(
            **params,
            random_state=42,
            verbose=0,
            allow_writing_files=False
        )
        
        model.fit(X_train_cv, y_train_cv)
        y_proba = model.predict_proba(X_val_cv)[:, 1]
        score = roc_auc_score(y_val_cv, y_proba)
        cv_scores.append(score)
    
    results.append({
        'params': params,
        'mean_test_score': np.mean(cv_scores),
        'std_test_score': np.std(cv_scores),
        'cv_scores': cv_scores
    })
    
    if (idx + 1) % 10 == 0:
        print(f"   ▶️  {idx + 1}/{n_iter} завершено")

search_time = time.time() - start_time
print(f"\n✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 4. ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)
# ----------------------------------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)

for key in param_distributions.keys():
    results_df[f'param_{key}'] = results_df['params'].apply(lambda x: x.get(key))

results_df['rank_test_score'] = results_df['mean_test_score'].rank(ascending=False).astype(int)

# 🔧 СНАЧАЛА создаём финальную модель
best_params = results_df.iloc[0]['params']
best_score = results_df.iloc[0]['mean_test_score']

best_cb = CatBoostClassifier(
    **best_params,
    random_state=42,
    verbose=0,
    allow_writing_files=False
)

best_cb.fit(X_bank_train, y_bank_train)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)")
print("=" * 70)

print(f"\n✅ Best ROC-AUC (CV): {best_score:.6f}")

# 🔧 ВСЕ ПАРАМЕТРЫ ПОИСКА
print(f"\n📋 Все лучшие параметры (из поиска):")
print("-" * 50)
for param, value in best_params.items():
    if param == 'scale_pos_weight' and value is not None:
        print(f"   {param:25s}: {value} ← вес миноритарного класса")
    elif param == 'scale_pos_weight' and value is None:
        print(f"   {param:25s}: {value} ← без взвешивания")
    else:
        print(f"   {param:25s}: {value}")

# 🔧 ВСЕ ПАРАМЕТРЫ МОДЕЛИ ПОСЛЕ ОБУЧЕНИЯ
print(f"\n📋 Фактические параметры обученной модели:")
print("-" * 50)
model_params = best_cb.get_params()
for param, value in model_params.items():
    print(f"   {param:25s}: {value}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_iterations', 
        'param_depth', 'param_learning_rate', 'param_scale_pos_weight']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 5. МЕТРИКИ НА ТЕСТЕ
# ----------------------------------------------------------------------------
y_pred = best_cb.predict(X_bank_test)
y_proba = best_cb.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print("\n" + "=" * 70)
print(" 📊 МЕТРИКИ НА ТЕСТЕ vs CV")
print("=" * 70)

print(f"\n⏱️  Время обучения: {search_time:.1f} сек")

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = best_score if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🌳 Структура модели:")
print(f"   • Деревьев: {best_cb.tree_count_}")
print(f"   • Глубина: {best_cb.get_params()['depth']}")
print(f"   • Learning Rate: {best_cb.get_params()['learning_rate']}")

# ----------------------------------------------------------------------------
# 6. АНАЛИЗ ПО ПАРАМЕТРАМ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 АНАЛИЗ ПО КЛЮЧЕВЫМ ПАРАМЕТРАМ")
print("=" * 70)

print(f"\n📈 По глубине деревьев:")
depth_stats = results_df.groupby('param_depth')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(depth_stats.to_string())

print(f"\n📈 По скорости обучения (LR):")
lr_stats = results_df.groupby('param_learning_rate')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(lr_stats.to_string())

print(f"\n📈 По scale_pos_weight (баланс классов):")
spw_stats = results_df.groupby('param_scale_pos_weight')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(spw_stats.to_string())

print(f"\n📈 По количеству итераций:")
iter_stats = results_df.groupby('param_iterations')['mean_test_score'].agg(['mean', 'std', 'count']).round(4)
print(iter_stats.to_string())

print("\n" + "=" * 70)
print(" ✅ CATBOOST + CUSTOM SEARCH ЗАВЕРШЕН!")
print("=" * 70)
  


 🔴 CATBOOST + CUSTOM RANDOMIZED SEARCH CV
⚙️  Параметры: 50 комбинаций, 5-Fold CV
⏱️  Ожидаемое время: ~2.1 мин

🔍 Генерация параметров...

🔍 Запуск кросс-валидации...
   📊 Размер данных: 8929 строк, 38 признаков
   ▶️  10/50 завершено
   ▶️  20/50 завершено
   ▶️  30/50 завершено
   ▶️  40/50 завершено
   ▶️  50/50 завершено

✅ Готово за 383.7 сек (6.40 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ (ПОЛНЫЙ СПИСОК)

✅ Best ROC-AUC (CV): 0.928412

📋 Все лучшие параметры (из поиска):
--------------------------------------------------
   iterations               : 200
   depth                    : 10
   learning_rate            : 0.05
   l2_leaf_reg              : 5
   subsample                : 0.8
   colsample_bylevel        : 0.8
   scale_pos_weight         : 1 ← вес миноритарного класса
   min_data_in_leaf         : 10

📋 Фактические параметры обученной модели:
--------------------------------------------------
   iterations               : 200
   learning_rate            : 0.05
   depth                 

In [ ]:
# ============================================================================
# 9. NAIVE BAYES (STOCK / DEFAULT) - ИСПРАВЛЕННЫЙ
# ============================================================================
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🔵 NAIVE BAYES (STOCK)")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. ВЫБОР ВЕРСИИ NAIVE BAYES
# ----------------------------------------------------------------------------
print("\n⚙️  Используем GaussianNB (непрерывные признаки)")

# ----------------------------------------------------------------------------
# 2. СОЗДАНИЕ МОДЕЛИ
# ----------------------------------------------------------------------------
nb_model = GaussianNB(
    var_smoothing=1e-9,
    priors=None
)

# ----------------------------------------------------------------------------
# 3. ОБУЧЕНИЕ
# ----------------------------------------------------------------------------
start_time = time.time()
nb_model.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

# ----------------------------------------------------------------------------
# 4. ПРЕДСКАЗАНИЯ
# ----------------------------------------------------------------------------
y_pred_nb = nb_model.predict(X_bank_test)
y_proba_nb = nb_model.predict_proba(X_bank_test)[:, 1]

# ----------------------------------------------------------------------------
# 5. МЕТРИКИ
# ----------------------------------------------------------------------------
metrics_nb = {
    'Accuracy': accuracy_score(y_bank_test, y_pred_nb),
    'Precision': precision_score(y_bank_test, y_pred_nb, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred_nb, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred_nb, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba_nb),
    'PR-AUC': average_precision_score(y_bank_test, y_proba_nb),
    'Time (s)': train_time
}

print(f"\n⏱️  Время обучения: {train_time:.3f} сек")

print(f"\n📊 МЕТРИКИ НА ТЕСТЕ:")
print("-" * 50)
for name, value in metrics_nb.items():
    if name != 'Time (s)':
        print(f"  {name:12s}: {value:.4f}")

# ----------------------------------------------------------------------------
# 6. ИНФОРМАЦИЯ О МОДЕЛИ
# ----------------------------------------------------------------------------
print(f"\n🔵 Информация о модели:")
print(f"   • Классы: {nb_model.classes_}")
print(f"   • Prior вероятности: {nb_model.class_prior_}")
print(f"   • Количество признаков: {nb_model.theta_.shape[1]}")


feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_score
}).sort_values('Importance', ascending=False).head(10)

for _, row in feature_importance.iterrows():
    bar = "█" * int(min(row['Importance'] * 100, 40))
    print(f"   {str(row['Feature']):<20s}: {row['Importance']:.4f} {bar}")

print("\n" + "=" * 70)
print(" ✅ NAIVE BAYES (STOCK) ЗАВЕРШЕН!")
print("=" * 70)

 🔵 NAIVE BAYES (STOCK)

⚙️  Используем GaussianNB (непрерывные признаки)

⏱️  Время обучения: 0.006 сек

📊 МЕТРИКИ НА ТЕСТЕ:
--------------------------------------------------
  Accuracy    : 0.7125
  Precision   : 0.7337
  Recall      : 0.6172
  F1-Score    : 0.6704
  ROC-AUC     : 0.7867
  PR-AUC      : 0.7621

🔵 Информация о модели:
   • Классы: [0 1]
   • Prior вероятности: [0.52615074 0.47384926]
   • Количество признаков: 38
   Feature_12          : 18.7542 ████████████████████████████████████████
   Feature_3           : 7.8592 ████████████████████████████████████████
   Feature_15          : 4.4071 ████████████████████████████████████████
   Feature_24          : 3.9622 ████████████████████████████████████████
   Feature_5           : 3.3942 ████████████████████████████████████████
   Feature_13          : 1.4129 ████████████████████████████████████████
   Feature_25          : 1.3123 ████████████████████████████████████████
   Feature_6           : 1.0105 █████████████████████

In [23]:
# ============================================================================
# 9. NAIVE BAYES + CUSTOM RANDOMIZED SEARCH CV
# ============================================================================
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)
import pandas as pd
import numpy as np
import time
import warnings
import random
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 🔵 NAIVE BAYES + CUSTOM RANDOMIZED SEARCH CV")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ----------------------------------------------------------------------------
# Naive Bayes имеет мало гиперпараметров, поэтому тестируем разные версии
param_distributions = {
    'model_type': ['gaussian', 'multinomial', 'bernoulli', 'complement'],
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],  # Только для Gaussian
    'alpha': [0.1, 0.5, 1.0, 2.0, 5.0],  # Для Multinomial/Bernoulli/Complement
    'fit_prior': [True, False],
    'class_prior': [None, [0.5, 0.5], [0.3, 0.7], [0.7, 0.3]],
}

n_iter = 40
cv_splits = 5

print(f"\n⚙️  Параметры: {n_iter} комбинаций, {cv_splits}-Fold CV")
print(f"⚠️  Тестируем 4 версии Naive Bayes")
print(f"⏱️  Ожидаемое время: ~{(n_iter * cv_splits * 0.2) / 60:.1f} мин")

# ----------------------------------------------------------------------------
# 2. ГЕНЕРАЦИЯ СЛУЧАЙНЫХ КОМБИНАЦИЙ
# ----------------------------------------------------------------------------
print("\n🔍 Генерация параметров...")

def generate_random_params(param_dist, n_iter, random_state=42):
    random.seed(random_state)
    params_list = []
    keys = list(param_dist.keys())
    
    for _ in range(n_iter):
        param_set = {key: random.choice(param_dist[key]) for key in keys}
        params_list.append(param_set)
    
    return params_list

random_params = generate_random_params(param_distributions, n_iter, random_state=42)

# ----------------------------------------------------------------------------
# 3. КАСТОМНАЯ КРОСС-ВАЛИДАЦИЯ
# ----------------------------------------------------------------------------
print("\n🔍 Запуск кросс-валидации...")

cv_strategy = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

results = []
start_time = time.time()

# Преобразуем в numpy для индексации
X_train_np = X_bank_train.values if hasattr(X_bank_train, 'values') else X_bank_train
y_train_np = y_bank_train.values if hasattr(y_bank_train, 'values') else y_bank_train

print(f"   📊 Размер данных: {X_train_np.shape[0]} строк, {X_train_np.shape[1]} признаков")

for idx, params in enumerate(random_params):
    cv_scores = []
    model_type = params.pop('model_type')
    
    for train_idx, val_idx in cv_strategy.split(X_train_np, y_train_np):
        X_train_cv, X_val_cv = X_train_np[train_idx], X_train_np[val_idx]
        y_train_cv, y_val_cv = y_train_np[train_idx], y_train_np[val_idx]
        
        # Создание модели в зависимости от типа
        if model_type == 'gaussian':
            model = GaussianNB(
                var_smoothing=params['var_smoothing'],
                priors=params['class_prior']
            )
        elif model_type == 'multinomial':
            model = MultinomialNB(
                alpha=params['alpha'],
                fit_prior=params['fit_prior'],
                class_prior=params['class_prior']
            )
        elif model_type == 'bernoulli':
            model = BernoulliNB(
                alpha=params['alpha'],
                fit_prior=params['fit_prior'],
                class_prior=params['class_prior']
            )
        elif model_type == 'complement':
            model = ComplementNB(
                alpha=params['alpha'],
                fit_prior=params['fit_prior'],
                class_prior=params['class_prior']
            )
        
        try:
            model.fit(X_train_cv, y_train_cv)
            y_proba = model.predict_proba(X_val_cv)[:, 1]
            score = roc_auc_score(y_val_cv, y_proba)
            cv_scores.append(score)
        except Exception as e:
            cv_scores.append(0.5)  # Провал — минимальный score
    
    results.append({
        'params': {**params, 'model_type': model_type},
        'mean_test_score': np.mean(cv_scores),
        'std_test_score': np.std(cv_scores),
        'cv_scores': cv_scores
    })
    
    if (idx + 1) % 10 == 0:
        print(f"   ▶️  {idx + 1}/{n_iter} завершено")

search_time = time.time() - start_time
print(f"\n✅ Готово за {search_time:.1f} сек ({search_time/60:.2f} мин)")

# ----------------------------------------------------------------------------
# 4. ЛУЧШИЕ ПАРАМЕТРЫ
# ----------------------------------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)

results_df['param_model_type'] = results_df['params'].apply(lambda x: x.get('model_type'))
results_df['param_var_smoothing'] = results_df['params'].apply(lambda x: x.get('var_smoothing'))
results_df['param_alpha'] = results_df['params'].apply(lambda x: x.get('alpha'))
results_df['param_class_prior'] = results_df['params'].apply(lambda x: str(x.get('class_prior')))

results_df['rank_test_score'] = results_df['mean_test_score'].rank(ascending=False).astype(int)

print("\n" + "=" * 70)
print(" 🏆 ЛУЧШИЕ ПАРАМЕТРЫ И ТОП-3")
print("=" * 70)

best_params = results_df.iloc[0]['params']
best_score = results_df.iloc[0]['mean_test_score']

print(f"\n✅ Best ROC-AUC (CV): {best_score:.6f}")
print(f"   Model Type: {best_params.get('model_type')}")
print(f"   Var Smoothing: {best_params.get('var_smoothing')}")
print(f"   Alpha: {best_params.get('alpha')}")
print(f"   Class Prior: {best_params.get('class_prior')}")

print(f"\n📋 Топ-3 комбинации:")
cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'param_model_type', 
        'param_var_smoothing', 'param_alpha', 'param_class_prior']
print(results_df[cols].head(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 5. АНАЛИЗ ПО ТИПАМ МОДЕЛЕЙ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 СРАВНЕНИЕ ВЕРСИЙ NAIVE BAYES")
print("=" * 70)

model_stats = results_df.groupby('param_model_type')['mean_test_score'].agg(['mean', 'std', 'count', 'max']).round(4)
print(f"\n📈 Производительность по типам моделей:")
print(model_stats.to_string())

best_model_type = model_stats.loc[model_stats['mean'].idxmax()]
print(f"\n🏆 Лучший тип модели: {model_stats['mean'].idxmax()} (Mean AUC: {best_model_type['mean']:.4f})")

# ----------------------------------------------------------------------------
# 6. ФИНАЛЬНАЯ МОДЕЛЬ И МЕТРИКИ
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print(" 📊 ФИНАЛЬНАЯ МОДЕЛЬ НА ВСЕЙ ТРЕНИРОВОЧНОЙ ВЫБОРКЕ")
print("=" * 70)

model_type = best_params.get('model_type')

if model_type == 'gaussian':
    best_nb = GaussianNB(
        var_smoothing=best_params.get('var_smoothing'),
        priors=best_params.get('class_prior')
    )
elif model_type == 'multinomial':
    best_nb = MultinomialNB(
        alpha=best_params.get('alpha'),
        fit_prior=best_params.get('fit_prior'),
        class_prior=best_params.get('class_prior')
    )
elif model_type == 'bernoulli':
    best_nb = BernoulliNB(
        alpha=best_params.get('alpha'),
        fit_prior=best_params.get('fit_prior'),
        class_prior=best_params.get('class_prior')
    )
elif model_type == 'complement':
    best_nb = ComplementNB(
        alpha=best_params.get('alpha'),
        fit_prior=best_params.get('fit_prior'),
        class_prior=best_params.get('class_prior')
    )

start_time = time.time()
best_nb.fit(X_bank_train, y_bank_train)
train_time = time.time() - start_time

y_pred = best_nb.predict(X_bank_test)
y_proba = best_nb.predict_proba(X_bank_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_bank_test, y_pred),
    'Precision': precision_score(y_bank_test, y_pred, zero_division=0),
    'Recall': recall_score(y_bank_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_bank_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_bank_test, y_proba),
    'PR-AUC': average_precision_score(y_bank_test, y_proba)
}

print(f"\n⏱️  Время обучения: {train_time:.1f} сек")

print(f"\n{'Метрика':<12} | {'CV Mean':<10} | {'Тест':<10} | {'Разница':<10}")
print("-" * 55)
for name, value in metrics.items():
    cv_val = best_score if name == 'ROC-AUC' else None
    cv_str = f"{cv_val:.6f}" if cv_val else "N/A"
    diff_str = f"{value - cv_val:+.6f}" if cv_val else "N/A"
    print(f"{name:<12} | {cv_str:<10} | {value:<10.6f} | {diff_str:<10}")

print(f"\n🔵 Информация о модели:")
print(f"   • Тип: {model_type}")
print(f"   • Классы: {best_nb.classes_}")
print(f"   • Prior вероятности: {getattr(best_nb, 'class_prior_', 'N/A')}")


print("\n" + "=" * 70)
print(" ✅ NAIVE BAYES + CV ЗАВЕРШЕН!")
print("=" * 70)

 🔵 NAIVE BAYES + CUSTOM RANDOMIZED SEARCH CV

⚙️  Параметры: 40 комбинаций, 5-Fold CV
⚠️  Тестируем 4 версии Naive Bayes
⏱️  Ожидаемое время: ~0.7 мин

🔍 Генерация параметров...

🔍 Запуск кросс-валидации...
   📊 Размер данных: 8929 строк, 38 признаков
   ▶️  10/40 завершено
   ▶️  20/40 завершено
   ▶️  30/40 завершено
   ▶️  40/40 завершено

✅ Готово за 0.6 сек (0.01 мин)

 🏆 ЛУЧШИЕ ПАРАМЕТРЫ И ТОП-3

✅ Best ROC-AUC (CV): 0.793844
   Model Type: bernoulli
   Var Smoothing: 1e-07
   Alpha: 5.0
   Class Prior: None

📋 Топ-3 комбинации:
 rank_test_score  mean_test_score  std_test_score param_model_type  param_var_smoothing  param_alpha param_class_prior
               2         0.793844        0.011822        bernoulli         1.000000e-07          5.0              None
               2         0.793844        0.011822        bernoulli         1.000000e-06          5.0              None
               2         0.793844        0.011822        bernoulli         1.000000e-08          5.0  

In [35]:
# ============================================================================
# 🎒 BAGGING ENSEMBLE
# ============================================================================
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, average_precision_score
import time

print("=" * 60)
print(" 🎒 BAGGING ENSEMBLE")
print("=" * 60)

# Bagging на основе RandomForest с лучшими параметрами
bagging = BaggingClassifier(
    estimator=RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

start = time.time()
bagging.fit(X_bank_train, y_bank_train)
y_pred = bagging.predict(X_bank_test)
y_proba = bagging.predict_proba(X_bank_test)[:, 1]

print(f"⏱️  Время: {time.time() - start:.2f} сек")
print(f"\n📊 МЕТРИКИ:")
print(f"  ROC-AUC  : {roc_auc_score(y_bank_test, y_proba):.4f}")
print(f"  PR-AUC   : {average_precision_score(y_bank_test, y_proba):.4f}")
print(f"  F1-Score : {f1_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Accuracy : {accuracy_score(y_bank_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"\n🌳 Деревьев: {len(bagging.estimators_)}")

# LinReg
#    C                   : 0.1
#    class_weight        : {0: 1, 1: 2}  ← вес миноритарного класса = 2
#    max_iter            : 1000
#    penalty             : l1
#    solver              : liblinear
#    tol                 : 0.0001
# DesTree
#     splitter                 : best
#    min_samples_split        : 50
#    min_samples_leaf         : 5
#    max_features             : None
#    max_depth                : 20
#    criterion                : entropy
#    class_weight             : balanced
# RF
#    n_estimators             : 500
#    min_samples_split        : 10
#    min_samples_leaf         : 1
#    max_features             : sqrt
#    max_depth                : None
#    class_weight             : balanced
#    bootstrap                : True
# ExtraTrees
#   n_estimators             : 300
#    min_samples_split        : 10
#    min_samples_leaf         : 1
#    max_features             : None
#    max_depth                : 30
#    class_weight             : None
#    bootstrap                : True
# LGBM
#     subsample                : 0.9
#    reg_lambda               : 0.5
#    reg_alpha                : 0
#    num_leaves               : 31
#    n_estimators             : 200
#    min_child_samples        : 10
#    max_depth                : -1 ← без ограничения глубины
#    learning_rate            : 0.05
#    colsample_bytree         : 0.7
#    class_weight             : None
# GRADIENT BOOSTING
#    subsample                : 1.0 ← аналог class_weight для дисбаланса
#    n_estimators             : 300
#    min_samples_split        : 50
#    min_samples_leaf         : 20
#    min_impurity_decrease    : 0.01
#    max_features             : 0.7
#    max_depth                : 7
# CatBoost
#    iterations               : 200
#    depth                    : 10
#    learning_rate            : 0.05
#    l2_leaf_reg              : 5
#    subsample                : 0.8
#    colsample_bylevel        : 0.8
#    scale_pos_weight         : 1 ← вес миноритарного класса
#    min_data_in_leaf         : 10

 🎒 BAGGING ENSEMBLE
⏱️  Время: 47.00 сек

📊 МЕТРИКИ:
  ROC-AUC  : 0.9191
  PR-AUC   : 0.8838
  F1-Score : 0.8579
  Accuracy : 0.8585
  Precision: 0.8182
  Recall   : 0.9017

🌳 Деревьев: 50


In [43]:
# ============================================================================
# 🔄 BLENDING ENSEMBLE (С OOF ПРЕДСКАЗАНИЯМИ — ИСПРАВЛЕННЫЙ)
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, average_precision_score
import lightgbm as lgb
from catboost import CatBoostClassifier
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print(" 🔄 BLENDING ENSEMBLE (С OOF)")
print("=" * 60)

# 1. Базовые модели с лучшими параметрами
models = [
    ('LinReg', LogisticRegression(
        C=0.1, class_weight={0: 1, 1: 2}, max_iter=1000,
        penalty='l1', solver='liblinear', tol=0.0001, random_state=42
    )),
    ('DesTree', DecisionTreeClassifier(
        splitter='best', min_samples_split=50, min_samples_leaf=5,
        max_features=None, max_depth=20, criterion='entropy',
        class_weight='balanced', random_state=42
    )),
    ('RF', RandomForestClassifier(
        n_estimators=500, min_samples_split=10, min_samples_leaf=1,
        max_features='sqrt', max_depth=None, class_weight='balanced',
        bootstrap=True, random_state=42, n_jobs=-1
    )),
    ('ExtraTrees', ExtraTreesClassifier(
        n_estimators=300, min_samples_split=10, min_samples_leaf=1,
        max_features=None, max_depth=30, class_weight=None,
        bootstrap=True, random_state=42, n_jobs=-1
    )),
    ('LGBM', lgb.LGBMClassifier(
        subsample=0.9, reg_lambda=0.5, reg_alpha=0, num_leaves=31,
        n_estimators=200, min_child_samples=10, max_depth=-1,
        learning_rate=0.05, colsample_bytree=0.7, class_weight=None,
        random_state=42, n_jobs=-1, verbose=-1
    )),
    ('GradBoost', GradientBoostingClassifier(
        subsample=1.0, n_estimators=300, min_samples_split=50,
        min_samples_leaf=20, min_impurity_decrease=0.01,
        max_features=0.7, max_depth=7, random_state=42
    )),
    ('CatBoost', CatBoostClassifier(
        iterations=200, depth=10, learning_rate=0.05, l2_leaf_reg=5,
        subsample=0.8, colsample_bylevel=0.8, scale_pos_weight=1,
        min_data_in_leaf=10, random_state=42, verbose=0,
        allow_writing_files=False
    )),
    ('NB', GaussianNB())
]

# 🔧 ИСПРАВЛЕНИЕ: Конвертируем в numpy для позиционной индексации
X_train_np = X_bank_train.values if hasattr(X_bank_train, 'values') else X_bank_train
y_train_np = y_bank_train.values if hasattr(y_bank_train, 'values') else y_bank_train

# 2. OOF предсказания для train (чтобы не было утечки)
print("\n🔄 Генерация OOF предсказаний...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X_train_np), len(models)))
test_preds = np.zeros((len(X_bank_test), len(models)))

start = time.time()
for i, (name, model) in enumerate(models):
    oof_fold_preds = np.zeros(len(X_train_np))
    test_fold_preds = np.zeros(len(X_bank_test))
    
    for train_idx, val_idx in cv.split(X_train_np, y_train_np):
        # 🔧 ТЕПЕРЬ РАБОТАЕТ: numpy массивы + позиционная индексация
        X_train_cv, X_val_cv = X_train_np[train_idx], X_train_np[val_idx]
        y_train_cv = y_train_np[train_idx]
        
        # Копируем модель для каждого фолда
        model_copy = type(model)(**model.get_params())
        model_copy.fit(X_train_cv, y_train_cv)
        
        oof_fold_preds[val_idx] = model_copy.predict_proba(X_val_cv)[:, 1]
        test_fold_preds += model_copy.predict_proba(X_bank_test)[:, 1] / cv.n_splits
    
    oof_preds[:, i] = oof_fold_preds
    test_preds[:, i] = test_fold_preds
    print(f"   ✅ {name} — OOF ROC-AUC: {roc_auc_score(y_train_np, oof_fold_preds):.4f}")

# 3. Мета-модель обучается на OOF предсказаниях (без утечки!)
print("\n📚 Обучение мета-модели...")
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(oof_preds, y_train_np)

# Финальные предсказания
y_proba = meta_model.predict_proba(test_preds)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

print(f"\n⏱️  Время: {time.time() - start:.2f} сек")
print(f"\n📊 МЕТРИКИ:")
print(f"  ROC-AUC  : {roc_auc_score(y_bank_test, y_proba):.4f}")
print(f"  PR-AUC   : {average_precision_score(y_bank_test, y_proba):.4f}")
print(f"  F1-Score : {f1_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Accuracy : {accuracy_score(y_bank_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"\n🔧 Базовых моделей: {len(models)}")

 🔄 BLENDING ENSEMBLE (С OOF)

🔄 Генерация OOF предсказаний...
   ✅ LinReg — OOF ROC-AUC: 0.8878
   ✅ DesTree — OOF ROC-AUC: 0.8930
   ✅ RF — OOF ROC-AUC: 0.9219
   ✅ ExtraTrees — OOF ROC-AUC: 0.9194
   ✅ LGBM — OOF ROC-AUC: 0.9281
   ✅ GradBoost — OOF ROC-AUC: 0.9252
   ✅ CatBoost — OOF ROC-AUC: 0.9284
   ✅ NB — OOF ROC-AUC: 0.7839

📚 Обучение мета-модели...

⏱️  Время: 43.91 сек

📊 МЕТРИКИ:
  ROC-AUC  : 0.9313
  PR-AUC   : 0.9026
  F1-Score : 0.8642
  Accuracy : 0.8670
  Precision: 0.8370
  Recall   : 0.8932

🔧 Базовых моделей: 8


In [45]:
# ============================================================================
# 📚 STACKING ENSEMBLE (С OOF — БЕЗ CATBOOST)
# ============================================================================
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, average_precision_score
import lightgbm as lgb
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print(" 📚 STACKING ENSEMBLE (С OOF)")
print("=" * 60)

# Базовые модели (уровень 1) с лучшими параметрами — БЕЗ CATBOOST
base_models = [
    ('LinReg', LogisticRegression(
        C=0.1, class_weight={0: 1, 1: 2}, max_iter=1000,
        penalty='l1', solver='liblinear', tol=0.0001, random_state=42
    )),
    ('DesTree', DecisionTreeClassifier(
        splitter='best', min_samples_split=50, min_samples_leaf=5,
        max_features=None, max_depth=20, criterion='entropy',
        class_weight='balanced', random_state=42
    )),
    ('RF', RandomForestClassifier(
        n_estimators=500, min_samples_split=10, min_samples_leaf=1,
        max_features='sqrt', max_depth=None, class_weight='balanced',
        bootstrap=True, random_state=42, n_jobs=-1
    )),
    ('ExtraTrees', ExtraTreesClassifier(
        n_estimators=300, min_samples_split=10, min_samples_leaf=1,
        max_features=None, max_depth=30, class_weight=None,
        bootstrap=True, random_state=42, n_jobs=-1
    )),
    ('LGBM', lgb.LGBMClassifier(
        subsample=0.9, reg_lambda=0.5, reg_alpha=0, num_leaves=31,
        n_estimators=200, min_child_samples=10, max_depth=-1,
        learning_rate=0.05, colsample_bytree=0.7, class_weight=None,
        random_state=42, n_jobs=-1, verbose=-1
    )),
    ('GradBoost', GradientBoostingClassifier(
        subsample=1.0, n_estimators=300, min_samples_split=50,
        min_samples_leaf=20, min_impurity_decrease=0.01,
        max_features=0.7, max_depth=7, random_state=42
    )),
    ('NB', GaussianNB())
]

# Мета-модель (уровень 2)
meta_model = LogisticRegression(
    C=0.1, class_weight={0: 1, 1: 2}, max_iter=1000,
    penalty='l1', solver='liblinear', tol=0.0001, random_state=42
)

# Стекинг с кросс-валидацией (OOF внутри)
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False  # Не добавлять исходные признаки
)

start = time.time()
stacking.fit(X_bank_train, y_bank_train)
y_pred = stacking.predict(X_bank_test)
y_proba = stacking.predict_proba(X_bank_test)[:, 1]

print(f"⏱️  Время: {time.time() - start:.2f} сек")
print(f"\n📊 МЕТРИКИ:")
print(f"  ROC-AUC  : {roc_auc_score(y_bank_test, y_proba):.4f}")
print(f"  PR-AUC   : {average_precision_score(y_bank_test, y_proba):.4f}")
print(f"  F1-Score : {f1_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Accuracy : {accuracy_score(y_bank_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_bank_test, y_pred, zero_division=0):.4f}")
print(f"\n🔧 Базовых моделей: {len(base_models)}")

 📚 STACKING ENSEMBLE (С OOF)
⏱️  Время: 10.85 сек

📊 МЕТРИКИ:
  ROC-AUC  : 0.9298
  PR-AUC   : 0.9011
  F1-Score : 0.8719
  Accuracy : 0.8701
  Precision: 0.8184
  Recall   : 0.9329

🔧 Базовых моделей: 7
